In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# ANALYSIS — physical_itens_venda_caixa
# Squad 3 — Arquitetura Medalhao
# Objetivo: cruzar KPIs Gold e gerar
#           insights de negocio
# Regra: le apenas tabelas Gold
#        nao recalcula, nao grava
# ══════════════════════════════════════

In [0]:

%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# tabelas Gold — unica fonte de dados

GOLD_KPI6_TABLE    = f"{TARGET_SCHEMA}.gold_kpi_receita_produto_loja_mes"
GOLD_KPI7_TABLE    = f"{TARGET_SCHEMA}.gold_kpi_top10_produtos_trimestre"
GOLD_KPI8_TABLE    = f"{TARGET_SCHEMA}.gold_kpi_mom_receita_loja"
GOLD_KPI9_TABLE    = f"{TARGET_SCHEMA}.gold_kpi_ticket_medio_loja"
GOLD_KPI4_TABLE    = f"{TARGET_SCHEMA}.gold_kpi_receita_loja_ano"
GOLD_KPI8ALT_TABLE = f"{TARGET_SCHEMA}.gold_kpi_mom_receita_tipo_pagamento"
GOLD_KPI10_TABLE   = f"{TARGET_SCHEMA}.gold_kpi_vendas_feriado_loja"

print("Tabelas Gold configuradas!")

In [0]:
# ler todas as tabelas Gold

from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    avg, round as spark_round,
    max as spark_max, min as spark_min,
    when, desc, asc, abs as spark_abs,
    first
)
from pyspark.sql.types import DoubleType, IntegerType

df_kpi6 = read_sql_table(spark, GOLD_KPI6_TABLE) \
    .withColumn("receita_total",
        col("receita_total").cast(DoubleType())) \
    .withColumn("qtd_total_vendida",
        col("qtd_total_vendida").cast(DoubleType())) \
    .withColumn("total_transacoes",
        col("total_transacoes").cast(IntegerType())) \
    .withColumn("preco_medio",
        col("preco_medio").cast(DoubleType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("mes",
        col("mes").cast(IntegerType()))

df_kpi7 = read_sql_table(spark, GOLD_KPI7_TABLE) \
    .withColumn("receita_total",
        col("receita_total").cast(DoubleType())) \
    .withColumn("qtd_total_vendida",
        col("qtd_total_vendida").cast(DoubleType())) \
    .withColumn("rank_receita",
        col("rank_receita").cast(IntegerType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("trimestre",
        col("trimestre").cast(IntegerType()))

df_kpi8 = read_sql_table(spark, GOLD_KPI8_TABLE) \
    .withColumn("receita_total",
        col("receita_total").cast(DoubleType())) \
    .withColumn("receita_mes_anterior",
        col("receita_mes_anterior").cast(DoubleType())) \
    .withColumn("crescimento_mom_pct",
        col("crescimento_mom_pct").cast(DoubleType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("mes",
        col("mes").cast(IntegerType()))

df_kpi9 = read_sql_table(spark, GOLD_KPI9_TABLE) \
    .withColumn("ticket_medio_valor",
        col("ticket_medio_valor").cast(DoubleType())) \
    .withColumn("ticket_medio_itens",
        col("ticket_medio_itens").cast(DoubleType())) \
    .withColumn("total_transacoes",
        col("total_transacoes").cast(IntegerType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("mes",
        col("mes").cast(IntegerType()))

df_kpi4 = read_sql_table(spark, GOLD_KPI4_TABLE) \
    .withColumn("receita_total_ano",
        col("receita_total_ano").cast(DoubleType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType()))

df_kpi8alt = read_sql_table(spark, GOLD_KPI8ALT_TABLE) \
    .withColumn("receita_total",
        col("receita_total").cast(DoubleType())) \
    .withColumn("crescimento_mom_receita_pct",
        col("crescimento_mom_receita_pct").cast(DoubleType())) \
    .withColumn("ano",
        col("ano").cast(IntegerType())) \
    .withColumn("mes",
        col("mes").cast(IntegerType()))

print("Gold KPIs carregados!")
print(f"   KPI 6  (receita produto/loja/mes) : {df_kpi6.count():,} linhas")
print(f"   KPI 7  (top 10 produtos/trimestre): {df_kpi7.count():,} linhas")
print(f"   KPI 8  (MoM receita/loja)         : {df_kpi8.count():,} linhas")
print(f"   KPI 9  (ticket medio/loja)        : {df_kpi9.count():,} linhas")
print(f"   KPI 4  (receita loja/ano)         : {df_kpi4.count():,} linhas")
print(f"   KPI 8ALT (MoM tipo pagamento)     : {df_kpi8alt.count():,} linhas")

In [0]:
# ══════════════════════════════════════
# INSIGHT 1
# Produtos ancora — consistentes no
# top 10 em todos os trimestres?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 1 — PRODUTOS ANCORA")
print("=" * 55)

from pyspark.sql.functions import countDistinct

# produtos que aparecem no top 10
# em mais de um trimestre
total_trimestres = df_kpi7 \
    .select("ano", "trimestre") \
    .distinct() \
    .count()

df_produtos_ancora = df_kpi7 \
    .groupBy("codigo_barras_produto") \
    .agg(
        countDistinct("trimestre").alias("trimestres_top10"),
        spark_round(
            spark_sum("receita_total"), 2
        ).alias("receita_total_acumulada"),
        spark_round(
            avg("rank_receita"), 1
        ).alias("rank_medio"),
        first("nome_loja").alias("loja_principal"),
    ) \
    .withColumn(
        "consistencia_pct",
        spark_round(
            col("trimestres_top10") /
            total_trimestres * 100, 1
        )
    ) \
    .orderBy(desc("trimestres_top10"), asc("rank_medio"))

print(f"\nTotal de trimestres analisados: {total_trimestres}")
print("\nProdutos mais consistentes no top 10:")
display(df_produtos_ancora.limit(15))

print("\nProdutos ancora por loja (presente em todos os trimestres):")
display(
    df_kpi7
    .groupBy("codigo_barras_produto", "id_loja", "nome_loja")
    .agg(
        countDistinct("trimestre").alias("trimestres_top10"),
        spark_round(avg("rank_receita"), 1).alias("rank_medio"),
        spark_round(spark_sum("receita_total"), 2).alias("receita_total"),
    )
    .filter(col("trimestres_top10") == total_trimestres)
    .orderBy("id_loja", asc("rank_medio"))
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 2
# Lojas com alto MoM de receita
# tambem tem alto ticket medio?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 2 — MoM RECEITA vs TICKET MEDIO")
print("=" * 55)

df_mom_medio = df_kpi8 \
    .filter(col("crescimento_mom_pct").isNotNull()) \
    .groupBy("id_loja", "nome_loja", "estado_loja") \
    .agg(
        spark_round(
            avg("crescimento_mom_pct"), 2
        ).alias("mom_medio_pct"),
        spark_round(
            avg("receita_total"), 2
        ).alias("receita_media_mensal"),
    )

df_ticket_medio = df_kpi9 \
    .groupBy("id_loja") \
    .agg(
        spark_round(
            avg("ticket_medio_valor"), 2
        ).alias("ticket_medio_geral"),
        spark_round(
            avg("ticket_medio_itens"), 2
        ).alias("itens_medio_geral"),
    )

df_insight2 = df_mom_medio \
    .join(df_ticket_medio, on="id_loja", how="left") \
    .withColumn(
        "perfil_crescimento",
        when(
            (col("mom_medio_pct") > 0) &
            (col("ticket_medio_geral") > 40),
            "Crescimento + Alto Ticket"
        )
        .when(
            (col("mom_medio_pct") > 0) &
            (col("ticket_medio_geral") <= 40),
            "Crescimento + Baixo Ticket"
        )
        .when(
            (col("mom_medio_pct") <= 0) &
            (col("ticket_medio_geral") > 40),
            "Queda + Alto Ticket"
        )
        .otherwise("Queda + Baixo Ticket")
    )

print("\nMoM medio vs Ticket medio por loja:")
display(
    df_insight2
    .select(
        "nome_loja",
        "estado_loja",
        "mom_medio_pct",
        "receita_media_mensal",
        "ticket_medio_geral",
        "itens_medio_geral",
        "perfil_crescimento"
    )
    .orderBy(desc("mom_medio_pct"))
)

print("\nDistribuicao de perfis:")
display(
    df_insight2
    .groupBy("perfil_crescimento")
    .agg(count("id_loja").alias("total_lojas"))
    .orderBy(desc("total_lojas"))
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 3
# Metodo de pagamento influencia
# o ticket medio da loja?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 3 — TIPO DE PAGAMENTO vs TICKET MEDIO")
print("=" * 55)

df_pagto_por_loja = df_kpi8alt \
    .groupBy("id_loja", "nome_loja", "tipo_pagamento") \
    .agg(
        spark_round(
            avg("receita_total"), 2
        ).alias("receita_media_mensal"),
        spark_round(
            avg("crescimento_mom_receita_pct"), 2
        ).alias("mom_medio_pct"),
        spark_sum("total_itens").alias("total_itens"),
    )

print("\nReceita media por tipo de pagamento:")
display(
    df_kpi8alt
    .groupBy("tipo_pagamento")
    .agg(
        spark_round(
            avg("receita_total"), 2
        ).alias("receita_media_mensal"),
        spark_round(
            avg("crescimento_mom_receita_pct"), 2
        ).alias("mom_medio_pct"),
        spark_sum("total_itens").alias("total_itens"),
        count("id_loja").alias("combinacoes_loja_mes")
    )
    .orderBy(desc("receita_media_mensal"))
)

print("\nTipo de pagamento dominante por loja:")
display(
    df_pagto_por_loja
    .orderBy("id_loja", desc("receita_media_mensal"))
)

print("\nTendencia MoM por tipo de pagamento:")
display(
    df_kpi8alt
    .filter(col("crescimento_mom_receita_pct").isNotNull())
    .groupBy("tipo_pagamento", "tendencia")
    .agg(count("id_loja").alias("ocorrencias"))
    .orderBy("tipo_pagamento", desc("ocorrencias"))
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 4
# Produtos mais vendidos variam
# entre lojas ou sao os mesmos?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 4 — HOMOGENEIDADE DE PRODUTOS ENTRE LOJAS")
print("=" * 55)

# produtos no top 1 de cada loja
df_top1_por_loja = df_kpi7 \
    .filter(col("rank_receita") == 1) \
    .select(
        "id_loja",
        "nome_loja",
        "estado_loja",
        "codigo_barras_produto",
        "receita_total",
        "ano",
        "trimestre"
    )

# quantas lojas diferentes tem o mesmo produto no top 1
df_produto_top1_lojas = df_top1_por_loja \
    .groupBy("codigo_barras_produto", "ano", "trimestre") \
    .agg(
        countDistinct("id_loja").alias("lojas_com_produto_top1"),
        spark_round(
            spark_sum("receita_total"), 2
        ).alias("receita_total_produto"),
    ) \
    .orderBy(desc("lojas_com_produto_top1"))

total_lojas_analisadas = df_kpi7 \
    .select("id_loja") \
    .distinct() \
    .count()

print(f"\nTotal de lojas analisadas: {total_lojas_analisadas:,}")
print("\nProdutos que lideram em mais lojas simultaneamente:")
display(df_produto_top1_lojas.limit(10))

print("\nTop 1 por loja — ha diversidade?")
display(
    df_top1_por_loja
    .groupBy("ano", "trimestre")
    .agg(
        countDistinct("codigo_barras_produto").alias("produtos_distintos_top1"),
        countDistinct("id_loja").alias("total_lojas")
    )
    .withColumn(
        "diversidade_pct",
        spark_round(
            col("produtos_distintos_top1") /
            col("total_lojas") * 100, 1
        )
    )
    .orderBy("ano", "trimestre")
)

In [0]:
# ══════════════════════════════════════
# INSIGHT 5
# Feriados aumentam ou diminuem
# o volume de vendas por loja?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 5 — IMPACTO DE FERIADOS NAS VENDAS")
print("=" * 55)

try:
    df_kpi10 = read_sql_table(spark, GOLD_KPI10_TABLE) \
        .withColumn("receita_total",
            col("receita_total").cast(DoubleType())) \
        .withColumn("total_itens",
            col("total_itens").cast(IntegerType())) \
        .withColumn("total_transacoes",
            col("total_transacoes").cast(IntegerType())) \
        .withColumn("ano",
            col("ano").cast(IntegerType())) \
        .withColumn("mes",
            col("mes").cast(IntegerType()))

    # comparar receita media em feriados vs dias normais
    df_comparacao = df_kpi10 \
        .groupBy("id_loja", "nome_loja", "venda_em_feriado") \
        .agg(
            spark_round(
                avg("receita_total"), 2
            ).alias("receita_media"),
            spark_round(
                avg("total_transacoes"), 0
            ).alias("transacoes_media"),
            spark_sum("total_itens").alias("total_itens"),
        )

    print("\nReceita media — Feriado vs Dia normal:")
    display(
        df_comparacao
        .groupBy("venda_em_feriado")
        .agg(
            spark_round(avg("receita_media"), 2).alias("receita_media_geral"),
            spark_round(avg("transacoes_media"), 0).alias("transacoes_media_geral"),
            spark_sum("total_itens").alias("total_itens"),
        )
    )

    print("\nImpacto por loja — feriado aumenta ou diminui?")
    df_pivot = df_comparacao \
        .groupBy("id_loja", "nome_loja") \
        .pivot("venda_em_feriado", ["true", "false"]) \
        .agg(spark_round(avg("receita_media"), 2)) \
        .withColumnRenamed("true", "receita_feriado") \
        .withColumnRenamed("false", "receita_dia_normal") \
        .withColumn(
            "impacto_feriado_pct",
            when(
                col("receita_dia_normal").isNotNull() &
                (col("receita_dia_normal") > 0),
                spark_round(
                    (col("receita_feriado") - col("receita_dia_normal")) /
                    col("receita_dia_normal") * 100, 2
                )
            ).otherwise(None)
        ) \
        .withColumn(
            "feriado_impacta",
            when(col("impacto_feriado_pct") > 0, "Aumenta vendas")
            .when(col("impacto_feriado_pct") < 0, "Diminui vendas")
            .otherwise("Sem impacto")
        )

    display(
        df_pivot
        .select(
            "nome_loja",
            "receita_feriado",
            "receita_dia_normal",
            "impacto_feriado_pct",
            "feriado_impacta"
        )
        .orderBy(desc("impacto_feriado_pct"))
    )

    print("\nFeriados com maior receita:")
    display(
        df_kpi10
        .filter(col("venda_em_feriado") == True)
        .groupBy("nome_feriado", "tipo_feriado")
        .agg(
            spark_round(
                spark_sum("receita_total"), 2
            ).alias("receita_total_feriado"),
            spark_sum("total_transacoes").alias("total_transacoes"),
        )
        .orderBy(desc("receita_total_feriado"))
    )

except Exception as e:
    print(f"Atencao: KPI feriados nao disponivel. {e}")

In [0]:
# ══════════════════════════════════════
# INSIGHT 6
# Lojas com maior receita tambem tem
# maior crescimento MoM?
# ══════════════════════════════════════

print("=" * 55)
print("INSIGHT 6 — RECEITA ALTA vs CRESCIMENTO MoM")
print("=" * 55)

df_receita_mensal_media = df_kpi6 \
    .groupBy("id_loja") \
    .agg(
        spark_round(
            avg("receita_total"), 2
        ).alias("receita_media_mensal"),
        spark_round(
            avg("preco_medio"), 2
        ).alias("preco_medio_produtos"),
        countDistinct("codigo_barras_produto").alias("mix_produtos"),
    )

df_mom_media = df_kpi8 \
    .filter(col("crescimento_mom_pct").isNotNull()) \
    .groupBy("id_loja", "nome_loja", "estado_loja") \
    .agg(
        spark_round(
            avg("crescimento_mom_pct"), 2
        ).alias("mom_medio_pct"),
    )

df_insight6 = df_mom_media \
    .join(df_receita_mensal_media, on="id_loja", how="left") \
    .join(
        df_kpi4.groupBy("id_loja").agg(
            spark_round(
                avg("receita_total_ano"), 2
            ).alias("receita_media_ano")
        ),
        on="id_loja",
        how="left"
    ) \
    .withColumn(
        "quadrante",
        when(
            (col("receita_media_ano") > df_kpi4.agg(
                avg("receita_total_ano")).collect()[0][0]) &
            (col("mom_medio_pct") > 0),
            "Alta receita + Crescendo"
        )
        .when(
            (col("receita_media_ano") > df_kpi4.agg(
                avg("receita_total_ano")).collect()[0][0]) &
            (col("mom_medio_pct") <= 0),
            "Alta receita + Estagnada"
        )
        .when(
            (col("receita_media_ano") <= df_kpi4.agg(
                avg("receita_total_ano")).collect()[0][0]) &
            (col("mom_medio_pct") > 0),
            "Baixa receita + Crescendo"
        )
        .otherwise("Baixa receita + Estagnada")
    )

print("\nQuadrante de performance — Receita vs MoM:")
display(
    df_insight6
    .select(
        "nome_loja",
        "estado_loja",
        "receita_media_ano",
        "mom_medio_pct",
        "mix_produtos",
        "preco_medio_produtos",
        "quadrante"
    )
    .orderBy(desc("receita_media_ano"))
)

print("\nDistribuicao por quadrante:")
display(
    df_insight6
    .groupBy("quadrante")
    .agg(
        count("id_loja").alias("total_lojas"),
        spark_round(
            avg("receita_media_ano"), 2
        ).alias("receita_media"),
        spark_round(
            avg("mom_medio_pct"), 2
        ).alias("mom_medio"),
    )
    .orderBy(desc("receita_media"))
)

In [0]:
# ══════════════════════════════════════
# RESUMO EXECUTIVO
# ══════════════════════════════════════

print("=" * 55)
print("RESUMO EXECUTIVO — ITENS VENDA CAIXA")
print("=" * 55)

receita_total = df_kpi6 \
    .agg(spark_round(spark_sum("receita_total"), 2)) \
    .collect()[0][0]

total_produtos = df_kpi6 \
    .select("codigo_barras_produto") \
    .distinct() \
    .count()

ticket_medio = df_kpi9 \
    .agg(spark_round(avg("ticket_medio_valor"), 2)) \
    .collect()[0][0]

itens_medio = df_kpi9 \
    .agg(spark_round(avg("ticket_medio_itens"), 2)) \
    .collect()[0][0]

produto_top1 = df_kpi7 \
    .filter(col("rank_receita") == 1) \
    .orderBy(desc("receita_total")) \
    .select("codigo_barras_produto") \
    .first()[0]

lojas_crescendo = df_kpi8 \
    .filter(
        col("crescimento_mom_pct").isNotNull() &
        (col("crescimento_mom_pct") > 0)
    ) \
    .select("id_loja") \
    .distinct() \
    .count()

total_lojas_mom = df_kpi8 \
    .filter(col("crescimento_mom_pct").isNotNull()) \
    .select("id_loja") \
    .distinct() \
    .count()

print(f"""
Visao Geral:
   Receita total             : R$ {receita_total:,.2f}
   Produtos distintos        : {total_produtos:,}
   Ticket medio por transacao: R$ {ticket_medio:,.2f}
   Itens medios por transacao: {itens_medio:,.1f}

Destaques:
   Produto maior receita     : {produto_top1}
   Lojas com MoM positivo    : {lojas_crescendo}/{total_lojas_mom}

Insights principais:
   INSIGHT 1 : Produtos ancora (consistentes top 10)
   INSIGHT 2 : MoM receita vs ticket medio
   INSIGHT 3 : Tipo de pagamento vs ticket medio
   INSIGHT 4 : Homogeneidade de produtos entre lojas
   INSIGHT 5 : Impacto de feriados nas vendas
   INSIGHT 6 : Quadrante receita alta vs crescimento MoM
""")